# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Zulfikar-CTRL-z/FlyRank-Internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**Binary Classification and Ranking.** I am framing Lane 2 (Refresh / Content Opportunity Scoring) as predicting whether a content item's organic search impressions will decline (`is_declining_label = 1` or `0`), so we can rank and prioritize pages for human review

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

The target proxy label is `is_declining_label`, derived from `trend_direction == 'down'`[4]. **Leakage Rule:** `trend_direction` and `trend_pct` are used to define this target label and must be excluded from model features to prevent feature leakage

In [16]:
import pandas as pd
try:
  df = pd.read_csv('data/raw/content_refresh_anonymized.csv')
except FileNotFoundError:
  df = pd.read_csv('https://raw.githubusercontent.com/Zulfikar-CTRL-z/FlyRank-Internship/main/data/raw/content_refresh_anonymized.csv')
df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)
print("Target Label Distribution (0 = Stable/Up, 1 = Down):")
print(df['is_declining_label'].value_counts(normalize=True))

Target Label Distribution (0 = Stable/Up, 1 = Down):
is_declining_label
1    0.542067
0    0.457933
Name: proportion, dtype: float64


## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Precision@50** (Precision in the top 50 ranked pages). Since the content review team has limited capacity, we want as many pages as possible in the top 50 flagged recommendations to actually be true declining pages.

In [17]:
total_declining = df['is_declining_label'].sum()
baseline_p50 = df['is_declining_label'].head(50).mean()
print(f"Total declining pages in sample: {total_declining}")
print(f"Baseline Precision@50 (first 50 rows): {baseline_p50:.2f}")

Total declining pages in sample: 16262
Baseline Precision@50 (first 50 rows): 0.68


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

**One row = One pseudonymized content item (** **content_id** **)** with observed search metrics, engagement data, and freshness signals

In [18]:
if 'is_declining_label' not in df.columns:
  df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)
print(f"Dataframe Shape: {df.shape}")
print("\nSample row (Unit of Analysis):")
display(df[['content_id', 'impressions_90d', 'ctr', 'trend_direction', 'is_declining_label']])

Dataframe Shape: (30000, 45)

Sample row (Unit of Analysis):


,content_id,impressions_90d,ctr,trend_direction,is_declining_label
0,content_304f48230142,3803,0.76,down,1
1,content_a1fb4e703a9e,15320,0.05,down,1
2,content_9aa793d4d895,12581,0.09,down,1
3,content_331d6c4de07b,11751,0.49,stable,0
4,content_d99b7a2d90ca,19140,0.13,down,1
...,...,...,...,...,...
29995,content_c322796023c8,1,0.00,new,0
29996,content_526572edb3fa,761,0.39,down,1
29997,content_38112bdd0c6e,6336,0.19,down,1
29998,content_ab26273a7e7a,154763,0.22,down,1


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A fixed hand-rule (if-statement) uses static thresholds like CTR < 0.5% or age > 180 days. Machine learning automatically learns complex, non-linear interactions across multiple signals.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.